In [1]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling

In [ ]:
cp_path = '../../dataset/region prediction/ChinaCP_2021.tif'
qa_path = '../../dataset/region prediction/ChinaCP-DA2021.tif'

src = rasterio.open(cp_path)
crop = src.read(1)
transform = src.transform
crs = src.crs

print("CRS:", crs)
print("Resolution:", src.res)

CRS: EPSG:32648
Resolution: (500.0, 500.0)


In [ ]:
qa_src = rasterio.open(qa_path)

qa_aligned = np.empty_like(crop, dtype=qa_src.read(1).dtype)

reproject(
    source=qa_src.read(1),
    destination=qa_aligned,
    src_transform=qa_src.transform,
    src_crs=qa_src.crs,
    dst_transform=transform,
    dst_crs=crs,
    resampling=Resampling.nearest
)

valid_mask = qa_aligned <= 2

print("Valid pixels:", valid_mask.sum())


Valid pixels: 34585575


In [ ]:
# ==========================
# look at the distribution of crop codes
# ==========================
unique_codes, counts = np.unique(crop, return_counts=True)

stats = pd.DataFrame({
    "CropCode": unique_codes,
    "PixelCount": counts
}).sort_values("PixelCount", ascending=False)

print("\nCrop code statistics:")
print(stats)


Crop code statistics:
      CropCode  PixelCount
0  -2147483648    71335293
6           17     3269751
7           27     2260240
4           15     1295289
3           14     1251526
1            0      650351
9          246      546601
2            3      176758
10         255      110171
5           16      109361
11         256       80412
8          245       21083


In [ ]:
# ==========================
# calculate pixel area in different units
# ==========================
pixel_width, pixel_height = src.res

pixel_area_m2 = abs(pixel_width * pixel_height)

pixel_area_ha = pixel_area_m2 / 10000
pixel_area_km2 = pixel_area_m2 / 1e6

print("\nPixel area:")
print(f"{pixel_area_m2:.2f} m²")
print(f"{pixel_area_ha:.6f} ha")
print(f"{pixel_area_km2:.8f} km²")


Pixel area:
250000.00 m²
25.000000 ha
0.25000000 km²


In [ ]:
# ==========================
# crop codes for maize and wheat
# ==========================
maize_codes = [14, 245, 246]
wheat_codes = [16, 246, 256]

maize_mask = np.isin(crop, maize_codes) & valid_mask
wheat_mask = np.isin(crop, wheat_codes) & valid_mask

In [7]:
# ==========================
# 像元统计
# ==========================
maize_pixels = maize_mask.sum()
wheat_pixels = wheat_mask.sum()

print("\nPixel counts:")
print("Maize pixels:", maize_pixels)
print("Wheat pixels:", wheat_pixels)



Pixel counts:
Maize pixels: 1746743
Wheat pixels: 732441


In [ ]:
# ==========================
# area calculation
# ==========================
maize_area_ha = maize_pixels * pixel_area_ha
maize_area_km2 = maize_pixels * pixel_area_km2

wheat_area_ha = wheat_pixels * pixel_area_ha
wheat_area_km2 = wheat_pixels * pixel_area_km2

In [ ]:
# ==========================
# output the results
# ==========================
print("\n========== Area Statistics ==========")

print(
    f"Maize area: "
    f"{maize_area_ha:,.0f} ha "
    f"({maize_area_km2:,.0f} km²)"
)

print(
    f"Wheat area: "
    f"{wheat_area_ha:,.0f} ha "
    f"({wheat_area_km2:,.0f} km²)"
)


========== Area Statistics ==========
Maize area: 43,668,575 ha (436,686 km²)
Wheat area: 18,311,025 ha (183,110 km²)


In [ ]:
# ==========================
# every crop code area calculation
# ==========================
rows = []

for code in unique_codes:

    mask = (crop == code) & valid_mask

    n = mask.sum()

    area_ha = n * pixel_area_ha

    area_km2 = n * pixel_area_km2

    rows.append([
        code,
        n,
        area_ha,
        area_km2
    ])

area_df = pd.DataFrame(
    rows,
    columns=[
        "CropCode",
        "Pixels",
        "Area_ha",
        "Area_km2"
    ]
)

area_df = area_df.sort_values(
    "Area_ha",
    ascending=False
)

print("\nArea by crop code:")
print(area_df)

# 保存
area_df.to_csv(
    "ChinaCP_crop_area_statistics.csv",
    index=False
)

print("\nSaved:")
print("ChinaCP_crop_area_statistics.csv")


Area by crop code:
      CropCode    Pixels      Area_ha    Area_km2
0  -2147483648  26293428  657335700.0  6573357.00
6           17   3043679   76091975.0   760919.75
7           27   1666080   41652000.0   416520.00
3           14   1190803   29770075.0   297700.75
4           15    938483   23462075.0   234620.75
1            0    556383   13909575.0   139095.75
9          246    545186   13629650.0   136296.50
5           16    107669    2691725.0    26917.25
2            3     94266    2356650.0    23566.50
11         256     79586    1989650.0    19896.50
10         255     59258    1481450.0    14814.50
8          245     10754     268850.0     2688.50

Saved:
ChinaCP_crop_area_statistics.csv


In [11]:
print(src.bounds)

BoundingBox(left=-2254124.036, bottom=2014256.0760000004, right=2716875.964, top=6093256.076)


In [12]:
print(src.res)

(500.0, 500.0)
